# Exploratory Data Analysis — TheLook Customer Analysis

**Scope:** Customer Analysis — who are our customers, how much value do they generate, how do they buy?

**Tables:** `users`, `orders`, `order_items`, `products`

**Data path:** `../data/raw/`

## 0. Imports & Load Data

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW = os.path.join("..", "data", "raw")

order_items = pd.read_csv(os.path.join(RAW, "order_items.csv"))
orders      = pd.read_csv(os.path.join(RAW, "orders.csv"))
products    = pd.read_csv(os.path.join(RAW, "products.csv"))
users       = pd.read_csv(os.path.join(RAW, "users.csv"))

In [ ]:
# Parse datetime columns
users['created_at'] = pd.to_datetime(users['created_at'], format='mixed', utc=True)

for col in ['created_at', 'shipped_at', 'delivered_at', 'returned_at']:
    order_items[col] = pd.to_datetime(order_items[col], format='mixed', utc=True)
    orders[col]      = pd.to_datetime(orders[col],      format='mixed', utc=True)

## 1. Dataset Overview

In [ ]:
tables = {
    "order_items": order_items,
    "orders":      orders,
    "products":    products,
    "users":       users,
}

print("Shape:")
for name, df in tables.items():
    print(f"  {name:<15} {df.shape[0]:>9,} rows  x  {df.shape[1]} cols")

print("\nDate range:")
for name, df in tables.items():
    if 'created_at' in df.columns:
        print(f"  {name:<15} {str(df['created_at'].min())[:10]}  →  {str(df['created_at'].max())[:10]}")

## 2. Data Quality

### 2.1 Missing Values (%)

In [ ]:
for name, df in tables.items():
    missing = (df.isnull().sum() / len(df) * 100).round(2)
    missing = missing[missing > 0]
    if not missing.empty:
        print(f"\n── {name} ──")
        print(missing.to_string())
    else:
        print(f"\n── {name} ── no missing values")

### 2.2 Duplicate Rows

In [ ]:
for name, df in tables.items():
    print(f"  {name:<15} {df.duplicated().sum():>5} duplicates")

## 3. Table: users

### 3.1 Age Distribution

In [ ]:
print(users['age'].describe().round(2))

plt.figure(figsize=(6, 4))
plt.boxplot(users['age'])
plt.title('Boxplot of Age')
plt.ylabel('Age')
plt.show()

users['age_group'] = pd.cut(
    users['age'],
    bins=[0, 18, 25, 35, 45, 55, float('inf')],
    right=False,
    labels=['<18', '18-24', '25-34', '35-44', '45-54', '55+']
)
print("\nAge group distribution:")
print(users['age_group'].value_counts().sort_index())

### 3.2 Gender & Country

In [ ]:
print("Gender (%):\n", users['gender'].value_counts(normalize=True).mul(100).round(2).to_string())
print("\nTop 10 Countries:\n", users['country'].value_counts().head(10).to_string())

### 3.3 Traffic Source (Acquisition)

In [ ]:
pct_ut = (users['traffic_source'].value_counts(normalize=True) * 100).round(2)

plt.figure(figsize=(8, 5))
plt.bar(pct_ut.index, pct_ut.values, color='steelblue')
plt.ylabel('Percentage (%)')
plt.xlabel('Traffic Source')
plt.title('User Acquisition by Traffic Source (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(pct_ut)

### 3.4 User Registration Over Time

In [ ]:
users['year']  = users['created_at'].dt.year
users['month'] = users['created_at'].dt.to_period('M')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

users.groupby('year').size().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('New Users by Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Users')
axes[0].tick_params(axis='x', rotation=0)

users.groupby('month').size().plot(ax=axes[1], color='steelblue')
axes[1].set_title('Monthly User Registrations')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Users')
axes[1].grid(False)

plt.tight_layout()
plt.show()

> User acquisition grows steadily year-on-year. 2024 appears lower because data only covers partial year.

## 4. Tables: orders & order_items

### 4.1 Order Status

In [ ]:
print("orders status (%):\n",
      orders['status'].value_counts(normalize=True).mul(100).round(2).to_string())

print("\norder_items status (%):\n",
      order_items['status'].value_counts(normalize=True).mul(100).round(2).to_string())

item_per_order = orders.groupby('order_id')['num_of_item'].sum().mean()
print(f"\nAvg items per order: {item_per_order:.2f}")

### 4.2 Conversion: Registered → Purchased

In [ ]:
total_users     = users['id'].nunique()
completed_orders = orders[orders['status'] == 'Complete']
buyers          = completed_orders['user_id'].nunique()
conversion_rate = buyers / total_users * 100

print(f"Total registered users: {total_users:,}")
print(f"Users who purchased:    {buyers:,}")
print(f"Conversion rate:        {conversion_rate:.1f}%")

### 4.3 Sale Price Distribution

In [ ]:
print(order_items['sale_price'].describe().round(2))

plt.figure(figsize=(8, 5))
plt.hist(order_items['sale_price'], bins=30, color='steelblue', edgecolor='white')
plt.title('Distribution of Sale Price')
plt.xlabel('Sale Price ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

> Right-skewed: most products are priced $0–$100, with a long tail of high-value items.

### 4.4 Revenue from Completed Orders

In [ ]:
completed     = order_items[order_items['status'] == 'Complete']
total_revenue = completed['sale_price'].sum()
aov           = completed.groupby('order_id')['sale_price'].sum().mean()

print(f"Total revenue (completed):  ${total_revenue:,.2f}")
print(f"Average Order Value (AOV):  ${aov:,.2f}")

## 5. Table: products

### 5.1 Category & Department

In [ ]:
print("Top Categories:\n", products['category'].value_counts().head(10).to_string())
print("\nDepartment split:\n", products['department'].value_counts().to_string())

### 5.2 Price Range by Category

In [ ]:
price_by_cat = products.groupby('category')['retail_price'].agg(['mean', 'median']).round(2)
print(price_by_cat.sort_values('mean', ascending=False).to_string())

---

## Summary of Key Findings

| Area | Observation |
|---|---|
| Users | Registration grows YoY; gender split ~50/50; age groups balanced |
| Conversion | Only ~27–28% of registered users ever complete a purchase |
| Orders | High cancel/return rate warrants investigation |
| Price | Right-skewed — majority of items under $100 |
| Products | Diverse category mix; Outerwear & Coats tend to be highest priced |